# RAGAS
To evaluate answer quality, I am going to generate question answer pairs using openai api


### 1. Environment and Library Setup
This block imports all the foundational libraries used for data handling (Pandas, LangChain), loading databases (ChromaDB), and testing (Ragas). It establishes the necessary ecosystem to build synthetic test questions and structure pipelines.


In [44]:
# Cell 1: Imports and setup
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import random
random.seed(69)



### 2. Path Configuration
Here, the local environment variables are loaded and the core working directory paths for the project are mapped, ensuring the script securely reads from the correct folders.


In [45]:
load_dotenv()

PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"

### 3. Loading the Vectorstore & Embeddings
This cell initializes the `HuggingFaceEmbeddings` and connects to the persistent local `ChromaDB`. We load the entire corpus of chunked algae literature into memory so that Ragas has factual grounding to draw from when generating test questions.


In [46]:
# Loads  chunks as LangChain documents
# Using the already-chunked data rather than raw PDFs since RAGAS should generate questions from what the pipeline actually retrieves



embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma(
    persist_directory=str(DATA_DIR / "chromadb"),
    collection_name="recursive_100",#typo of 1000
    embedding_function=embedding_model
)

# Get all documents from the collection
all_docs = vectorstore.get()
print(f"Loaded {len(all_docs['documents'])} chunks")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3008.47it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 31741 chunks


### 4. Sampling Documents for Test Generation
The full database contains over 30,000 chunks, which is too large and expensive for test generation. This block converts the raw chunks into LangChain `Document` format and samples 200 random chunks. This provides a diverse but manageable subset representing the real retrieval space.


### 5. LLM Utility Imports
This brief setup prepares the language model factories required by the Ragas generators, linking OpenAI or alternative models to the testing tools.


### 6. Synthetic Testset Generation
Here, the `gpt-4o-mini` model is configured as the generative engine to create synthetic question-answer pairs based on the sampled documents. The `TestsetGenerator` creates 50 test cases, forming a robust golden dataset mimicking real-world queries. The results are saved as a CSV.


### 7. Preview the Testset
A simple utility to preview the generated dataframe of test questions.


In [47]:
#df.head()

# Formal RAGAS Evaluation (Baseline vs. Hybrid)
This section runs the official quantitative comparison between the baseline (vector-only) and hybrid pipelines.
It tests both the General testset and the Multi-hop testset separately.

### 8. Pipeline Generation Tools Setup
This is the starting point for the automated comparison. This block connects the notebook to your local repository modules, importing the local `generate_answer` and `build_context` mechanics used by your hybrid system in production.


In [48]:
import os
import sys

import pandas as pd
from datasets import Dataset
from openai import OpenAI

from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]  

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import TOP_K_RETRIEVAL
from generation.generate import build_context, generate_answer

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### 9. Knowledge Graph Retrieval Setup
This block initializes connection to the Neo4j Graph Database. Crucially, it defines `expand_from_chunks()`, an advanced Cypher retriever. It takes vector chunks, explores their semantic relationships, filters out 'hub' nodes using degree penalty limits, and returns highly specific biological triplets. This step is the defining feature of your Hybrid GraphRAG pipeline.


In [49]:
# Hybrid Database Connections
from neo4j import GraphDatabase
from config import NEO4J_URI, NEO4J_USER
AUTH = (NEO4J_USER, "graphrag")
driver = GraphDatabase.driver(NEO4J_URI, auth=AUTH)

def expand_from_chunks(chunk_ids, driver, max_triplets=40):
    cypher = """
    MATCH (c:Chunk) WHERE c.chunk_id IN $chunk_ids
    MATCH (c)-[:MENTIONS]->(entity)
    
    // Degree penalty graph hub filter (from hybrid notebook)
    WITH entity
    WHERE size([(entity)-[]-() | 1]) < 100
    
    MATCH (entity)-[r]->(neighbor)
    WHERE type(r) IN ['FOUND_IN','PRODUCES','STUDIED_WITH',
                      'IDENTIFIED_BY','BELONGS_TO','AFFECTS','CONTAINS']
      AND r.confidence >= 0.7
    RETURN DISTINCT
        entity.name  AS subject,
        type(r)      AS predicate,
        neighbor.name AS object,
        r.confidence AS confidence
    ORDER BY r.confidence DESC
    LIMIT $max_triplets
    """
    with driver.session() as session:
        result = session.run(cypher, chunk_ids=chunk_ids, max_triplets=max_triplets)
        return [(r["subject"], r["predicate"], r["object"], r["confidence"])
                for r in result]

### 10. Configure DeepSeek Judge & Ragas Metrics
Rather than using standard automated metrics, this script configures **DeepSeek-Chat** via its API as the evaluator 'Judge'. An advanced 8k token limit is configured to handle large graph context sizes. It imports the 4 canonical metrics (Faithfulness, Answer Relevancy, Precision, Recall) initialized as classes to comply with modern `v0.4+` Ragas standards.


In [50]:
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
import os
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings
from ragas.embeddings.base import embedding_factory


deepseek_client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)
evaluator_llm = llm_factory(
    "deepseek-chat",
    provider="openai",
    client=deepseek_client,
    max_tokens=8192,  # DeepSeek-chat hard cap; reasoner goes to 64K
)
evaluator_embeddings = embedding_factory('openai', model='text-embedding-ada-002', client=openai_client, interface='modern')
# this is obsolete LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))



metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm),
]

### 11. Pipeline Executor Logic
The `run_eval_dataset` function simulates the entire RAG pipeline query-by-query over the testset. It takes the specific pipeline type (`baseline` vs `hybrid`). 
- **Baseline**: Merely retrieves chunks from Vectorstore.
- **Hybrid**: Retrieves vector chunks AND passes them through `expand_from_chunks` to attach biological triplet contexts.

It dynamically constructs the combined context and outputs a HuggingFace `Dataset` format expected by the evaluator.


In [51]:
from tqdm.auto import tqdm

def run_eval_dataset(dataset_path, pipeline_type):
    print(f"\n== Evaluating {pipeline_type.upper()} on {Path(dataset_path).name} ==")
    testset_df = pd.read_csv(dataset_path)
    data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    
    retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K_RETRIEVAL})
    
    for i, row in tqdm(testset_df.iterrows(), total=len(testset_df)):
        query = row["user_input"]
        ground_truth = str(row.get("reference", ""))
        
        # Retrieval
        entry_docs = retriever.invoke(query)
        entry_chunk_ids = [d.metadata.get("chunk_id") for d in entry_docs]
        vector_context, contexts_list = build_context([(1.0, doc) for doc in entry_docs])
        
        if pipeline_type == "baseline":
            final_context = vector_context
        else:
            triplets = expand_from_chunks(entry_chunk_ids, driver)
            triplet_lines = [f"- {s} {p} {o} (confidence: {c:.2f})" for s, p, o, c in triplets]
            final_context = vector_context + "\n\nRelated knowledge graph facts:\n" + "\n".join(triplet_lines)
            if triplet_lines:
                contexts_list.append("Related knowledge graph facts:\n" + "\n".join(triplet_lines))
            
        answer = generate_answer(query, final_context, openai_client)
        
        data["question"].append(query)
        data["answer"].append(answer)
        data["contexts"].append(contexts_list)
        data["ground_truth"].append(ground_truth)
        
    return Dataset.from_dict(data)

### 12. Evaluation Paths

Resolve the testset and output directories used by both pipelines.

In [52]:
from ragas import evaluate
from datasets import load_from_disk
REPO_ROOT = Path().resolve().parents[1]       # project/  for data and outputs

general_path = REPO_ROOT / "outputs" / "ragas_testset.csv"

ds = load_from_disk(str(REPO_ROOT / "outputs" / "baseline_ds"))
print(type(ds), len(ds), ds.column_names)


<class 'datasets.arrow_dataset.Dataset'> 51 ['question', 'answer', 'contexts', 'ground_truth']


### 13. Manual Metric Scoring

The installed `ragas` version ships two incompatible metric class hierarchies: the legacy `Metric` base class that `evaluate()` checks against, and the `BaseMetric` / `SimpleBaseMetric` tree that the modern `ragas.metrics.collections` metrics actually inherit from. The isinstance check inside `evaluate()` rejects the collections metrics, and monkey-patching past it breaks on `required_columns`. Rather than fight the framework, we score each sample directly through `single_turn_ascore`, which is the method `evaluate()` would call internally anyway. This produces identical numbers while sidestepping the broken dispatcher.

In [53]:
import asyncio
import nest_asyncio
from ragas.dataset_schema import SingleTurnSample
from tqdm.auto import tqdm

nest_asyncio.apply()

async def score_dataset(ds_hf, metrics):
    """Score every row in a HuggingFace Dataset against every metric.

    The ragas collections metrics each take a different subset of fields.
    We build a dispatch table mapping metric class names onto the kwargs
    they want. Simple, explicit, debuggable.
    """
    results = {m.name: [] for m in metrics}

    for i in tqdm(range(len(ds_hf)), desc="Scoring"):
        row = ds_hf[i]
        user_input = row["question"]
        response = row["answer"]
        retrieved_contexts = list(row["contexts"])
        reference = row["ground_truth"]

        for m in metrics:
            cls_name = type(m).__name__
            try:
                if cls_name == "Faithfulness":
                    result = await m.ascore(
                        user_input=user_input,
                        response=response,
                        retrieved_contexts=retrieved_contexts,
                    )
                elif cls_name == "AnswerRelevancy":
                    result = await m.ascore(
                        user_input=user_input,
                        response=response,
                    )
                elif cls_name.startswith("ContextPrecision"):
                    result = await m.ascore(
                        user_input=user_input,
                        reference=reference,
                        retrieved_contexts=retrieved_contexts,
                    )
                elif cls_name == "ContextRecall":
                    result = await m.ascore(
                        user_input=user_input,
                        retrieved_contexts=retrieved_contexts,
                        reference=reference,
                    )
                else:
                    raise ValueError(f"No dispatch rule for metric class {cls_name}")

                score = result.value if hasattr(result, "value") else result
                results[m.name].append(score)
            except Exception as e:
                print(f"Row {i} metric {m.name} failed: {e}")
                results[m.name].append(None)

    return results

In [54]:
from openai import AsyncOpenAI
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import AnswerRelevancy

async_openai_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

evaluator_embeddings = embedding_factory(
    "openai",
    model="text-embedding-3-small",
    client=async_openai_client,
    interface="modern",
)

metrics[1] = AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)

print("Fixed. metrics[1] is now:", metrics[1])

Fixed. metrics[1] is now: AnswerRelevancy(name='answer_relevancy', allowed_values=(0.0, 1.0))


### 14. Score the Baseline Dataset

The baseline dataset `ds` from the previous generation run is still in kernel memory, so there is no need to regenerate answers. We score it in place and write the result to CSV.

In [57]:
# Score the baseline ds that is already in memory from the earlier generation run.
# Do NOT rerun run_eval_dataset here  that would burn another round of API calls.
#save
def save_scored_dataset(ds_hf, scores, output_path):
    """Attach metric scores as columns onto the dataset and write to CSV."""
    out = ds_hf.to_pandas()
    for col, vals in scores.items():
        out[col] = vals
    out.to_csv(output_path, index=False)
    print(f"Wrote {len(out)} rows to {output_path}")
    print(out[list(scores.keys())].mean(numeric_only=True))
    return out

baseline_scores = asyncio.get_event_loop().run_until_complete(
    score_dataset(ds, metrics)
)
baseline_df = save_scored_dataset(
    ds,
    baseline_scores,
    REPO_ROOT / "outputs" / "baseline_eval_general.csv",
)

Scoring: 100%|██████████| 51/51 [1:03:23<00:00, 74.58s/it]

Wrote 51 rows to C:\Users\filip\Desktop\Thesis\project\outputs\baseline_eval_general.csv
faithfulness         0.744721
answer_relevancy     0.596464
context_precision    0.312282
context_recall       0.455556
dtype: float64


### 15. Generate and Score the Hybrid Pipeline

The hybrid run needs fresh generation because it routes each query through the Neo4j one-hop expansion. We reuse the same scorer so both CSVs are strictly comparable.

In [58]:
ds_hybrid = run_eval_dataset(general_path, "hybrid")

hybrid_scores = asyncio.get_event_loop().run_until_complete(
    score_dataset(ds_hybrid, metrics)
)
hybrid_df = save_scored_dataset(
    ds_hybrid,
    hybrid_scores,
    REPO_ROOT / "outputs" / "hybrid_eval_general.csv",
)


== Evaluating HYBRID on ragas_testset.csv ==


Scoring: 100%|██████████| 51/51 [1:06:59<00:00, 78.81s/it]

Wrote 51 rows to C:\Users\filip\Desktop\Thesis\project\outputs\hybrid_eval_general.csv
faithfulness         0.771941
answer_relevancy     0.655677
context_precision    0.333224
context_recall       0.482680
dtype: float64


### 16. Baseline vs Hybrid Summary

A quick side-by-side of the per-metric means. Positive `delta` values indicate the KG expansion improved that metric.

In [59]:
metric_cols = [m.name for m in metrics]

summary = pd.DataFrame({
    "baseline": baseline_df[metric_cols].mean(numeric_only=True),
    "hybrid":   hybrid_df[metric_cols].mean(numeric_only=True),
})
summary["delta"] = summary["hybrid"] - summary["baseline"]
print(summary.round(4))

                   baseline  hybrid   delta
faithfulness         0.7447  0.7719  0.0272
answer_relevancy     0.5965  0.6557  0.0592
context_precision    0.3123  0.3332  0.0209
context_recall       0.4556  0.4827  0.0271


# Fixes